# Exp D35 - TabPFN-3.5 na Colab T4 GPU-u (jednokratni, nekomercijalni eksperiment)
Potrebno: Runtime > Change runtime type > **T4 GPU**. Ćelije izvršavaj redom.
Licenca težina: `tabpfn-3-5-license-v1.0` (samo research/evaluation). Rezultati se **nikad** ne koriste u produkciji.
Težine se učitavaju isključivo iz lokalne datoteke (`model_path`); hosted API / Prior Labs klijent se ne koristi, `TABPFN_TOKEN` se briše, a SVAKI izlazni mrežni promet osim loopbacka (127.0.0.1, ::1, localhost, AF_UNIX) je blokiran tijekom fit/predict (allowlist).

## 1. Provjera GPU-a

In [ ]:
import subprocess, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'nvidia-smi: nema izlaza')
import torch
if not torch.cuda.is_available():
    raise SystemExit('NEMA GPU-a. Runtime > Change runtime type > T4 GPU, pa ponovno pokreni od vrha.')
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)
TORCH_BEFORE = torch.__version__

## 2. Upload zipa i raspakiranje
Upload `exp_d35_colab.zip` (skripte + 3 anonimizirana fajla).

In [ ]:
import os, zipfile
from google.colab import files
ROOT = '/content/expd35'
os.makedirs(ROOT, exist_ok=True)
up = files.upload()
name = next(n for n in up if n.endswith('.zip'))
with zipfile.ZipFile(name) as z:
    z.extractall(ROOT)
os.makedirs(f'{ROOT}/out', exist_ok=True)
for r, _, fs in os.walk(ROOT):
    for f in fs:
        print(os.path.join(r, f), os.path.getsize(os.path.join(r, f)))

## 3. Instalacija (tabpfn 10.0.0)
`tabpfn` traži `torch>=2.5`; ako Colab ima stariji torch, pip će ga nadograditi pa treba **Runtime > Restart session** i ponoviti od ćelije 1.

In [ ]:
!pip install -q tabpfn==10.0.0
import importlib.metadata as md
for p in ['tabpfn', 'torch', 'numpy', 'pandas', 'scikit-learn', 'pyarrow', 'safetensors', 'huggingface-hub']:
    print(p, md.version(p))
if md.version('torch') != TORCH_BEFORE.split('+')[0] and md.version('torch') != TORCH_BEFORE:
    print('\n!!! UPOZORENJE: torch se promijenio (' + TORCH_BEFORE + ' -> ' + md.version('torch') + '). '
          'Napravi Runtime > Restart session i ponovno pokreni od ćelije 1 (pip će tada biti no-op).')

## 4. Težine s Hugging Facea + SHA256
`Prior-Labs/tabpfn_3_5`, datoteka `tabpfn-v3.5-20260909.safetensors` (nije gated, bez tokena). Prvi put se samo ispisuje veličina i SHA256 - upiši ga u `EXPECTED_SHA` za sljedeće pokretanje. Ime datoteke mora sadržavati `v3.5` (tabpfn iz imena određuje verziju modela).

In [ ]:
import hashlib, os
from huggingface_hub import hf_hub_download
EXPECTED_SHA = ''   # upisati nakon prvog ispisa (ako je prazno, samo se ispisuje)
WEIGHTS_FILE = 'tabpfn-v3.5-20260909.safetensors'   # Fast: 'tabpfn-v3.5-fast-20260909.safetensors'
os.environ.pop('TABPFN_TOKEN', None)
ckpt = hf_hub_download('Prior-Labs/tabpfn_3_5', WEIGHTS_FILE, local_dir=f'{ROOT}/weights', token=False)
h = hashlib.sha256()
with open(ckpt, 'rb') as f:
    for b in iter(lambda: f.read(1 << 20), b''):
        h.update(b)
print(ckpt, os.path.getsize(ckpt), 'B')
print('SHA256:', h.hexdigest())
if EXPECTED_SHA:
    if h.hexdigest().lower() != EXPECTED_SHA.lower():
        os.remove(ckpt)
        raise SystemExit('SHA256 NE ODGOVARA - težine obrisane, ne nastavljaj.')
    os.environ['TABPFN35_EXPECTED_SHA'] = EXPECTED_SHA
    print('SHA256 OK')
else:
    print('EXPECTED_SHA je prazan - hash nije provjeren (prvo pokretanje).')
os.environ['TABPFN35_CKPT'] = ckpt

## 5. Provjera podataka (dry-run, bez modela)

In [ ]:
!cd {ROOT} && python scripts/run_walkforward_tabpfn35.py --workdir {ROOT}/ws --folds 1,2,3,4 --variants D-Bprime,D-Bprime-par --context-rows full --dry-run

## 6. Provjera da nema mrežnog kontakta s Prior Labs
Guard (`tabpfn35_guard.py`) blokira i bilježi svaki izlazni mrežni pokušaj osim loopbacka (DNS, connect, sendto; i po IP adresi, i telemetrija), `HF_HUB_OFFLINE=1`, `TABPFN_TOKEN` je obrisan. Ćelija radi samoprovjeru guarda i mali sintetički fit/predict. Ako paket pokuša kontakt, skripta staje s porukom `STOP` i **ne smije se nastaviti**.

In [ ]:
import subprocess, sys
for cmd in (['python', 'scripts/tabpfn35_guard.py'],
            ['python', 'scripts/tabpfn35_smoke_timing.py', '--synthetic', '--device', 'cuda']):
    p = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    print(p.stdout, p.stderr[-3000:])
    if p.returncode != 0:
        raise SystemExit('STOP: provjera mreže/guarda nije prošla - ne nastavljaj (vidi ispis iznad).')
print('Provjera mreže PROŠLA.')

## 7. TIMING - fold 4, D-Bprime, puni kontekst (~24.5k redova), n_test 2000, seed 42
Ispisuje procjenu `MIN_PER_RUN` (fit + predikcija kalibracijskog isječka i cijelog test seta). **Cilj: ispod 30 min po pokretanju.** Ako je više, u ćeliji 4 postavi `WEIGHTS_FILE = 'tabpfn-v3.5-fast-20260909.safetensors'` i ponovi 4, 6 i 7.

In [ ]:
import re, subprocess
p = subprocess.run(['python', 'scripts/tabpfn35_smoke_timing.py', '--workdir', f'{ROOT}/ws', '--fold', '4',
                    '--variant', 'D-Bprime', '--n-test', '2000', '--seed', '42', '--device', 'cuda'],
                   cwd=ROOT, capture_output=True, text=True)
print(p.stdout, p.stderr[-3000:])
if p.returncode != 0:
    raise SystemExit('Timing nije uspio (vidi ispis iznad).')
MIN_PER_RUN = float(re.search(r'MIN_PER_RUN=([0-9.]+)', p.stdout).group(1))
print(f'\n>>> {MIN_PER_RUN:.1f} min po pokretanju (procjena). Glavni dio: 6 pokretanja.')
if MIN_PER_RUN > 30:
    print('>>> PREKO 30 min: predlaže se Fast checkpoint (tabpfn-v3.5-fast-20260909.safetensors).')

## 8. GLAVNI eksperiment (checkpoint / resume)
Svako pokretanje sprema u `out/<run_id>/` i preskače se ako već postoji `DONE`. Nakon prekida ponovno pokreni ćelije 1-4 (i ovu; podaci ostaju u `/content/expd35` ako se sesija nije resetirala, inače ponovi upload).
Pokretanja (seed 42): fold 4 D-Bprime puni kontekst, fold 4 D-Bprime-par puni kontekst, fold 4 D-Bprime kontekst 10k (kontrola: učinak modela vs veličine konteksta), te walk-forward foldovi 1-3 D-Bprime puni kontekst.

In [ ]:
import json, os, subprocess, time, sys
TIMING_OK = False   # postavi True tek kad je timing iz ćelije 7 prihvatljiv (< 30 min)
RUN_WALKFORWARD = True
if not TIMING_OK:
    raise SystemExit('Prvo pogledaj timing (ćelija 7) pa postavi TIMING_OK = True.')

SEED = 42
runs = [(4, 'D-Bprime', 'full'), (4, 'D-Bprime-par', 'full'), (4, 'D-Bprime', '10000')]
if RUN_WALKFORWARD:
    runs += [(k, 'D-Bprime', 'full') for k in (1, 2, 3)]

def run_id(k, v, c):
    return f'f{k}_{v}__ctx{c}__s{SEED}'

t_all = time.time()
for i, (k, v, c) in enumerate(runs, 1):
    rid = run_id(k, v, c)
    od = f'{ROOT}/out/{rid}'
    if os.path.exists(f'{od}/DONE'):
        print(f'[{i}/{len(runs)}] skip {rid} (gotovo)')
        continue
    os.makedirs(od, exist_ok=True)
    print(f'[{i}/{len(runs)}] {rid}', flush=True)
    t0 = time.time()
    cmd = [sys.executable, 'scripts/run_walkforward_tabpfn35.py', '--workdir', f'{ROOT}/ws', '--variants', v,
           '--context-rows', c, '--seeds', str(SEED), '--folds', str(k), '--device', 'cuda', '--out-dir', od]
    p = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with open(f'{od}/log.txt', 'w', encoding='utf-8') as lg:
        for line in p.stdout:
            print(line, end=''); lg.write(line)
    if p.wait() != 0:
        raise SystemExit(f'Pokretanje {rid} nije uspjelo (vidi ispis iznad; STOP znači pokušaj vanjskog kontakta). Ponovno pokretanje ćelije nastavlja od ovdje.')
    json.dump({'run_id': rid, 'minutes': round((time.time() - t0) / 60, 1)}, open(f'{od}/DONE', 'w'))
    print(f'    {rid}: {(time.time() - t0) / 60:.1f} min (ukupno {(time.time() - t_all) / 60:.0f} min)', flush=True)
print('SVE GOTOVO')

## 9. Zipanje rezultata i preuzimanje

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/exp_d35_rezultati', 'zip', f'{ROOT}/out')
print(os.path.getsize('/content/exp_d35_rezultati.zip'), 'B')
files.download('/content/exp_d35_rezultati.zip')